# Two-Tower Deep Neural Network (DNN)

Goal:
* understand how to train 2 tower dnn

## Introduction

This notebook introduces the Two-Tower architecture, commonly used in modern recommendation systems.

We train a two-tower model for recommending movies to users using the MovieLens dataset. The model learns two embeddings: user embeddings and movie embeddings, by learning interaction patterns between users and movie ratings.

Finally, cosine distance is used to compute similarity scores between users and movies.

```
User Features ----> User Tower ----\
                                    |--> dot product --> Score
Movie Features --> Movie Tower ----/
```

**The goal is to learn how to define and train a two-tower DNN, rather than to train a highly optimized recommendation system.**

The goal is not discuss cold start, imbalance data, recommendation metrics like precision@k, recall@k, diversity and ect.

Two towers DNN has many applications:

1. Recommended systems
    * User ↔ Item matching (movies, products, videos)
    * **Job and candidate matching**: Resume ↔ Job description matching
1. **Search engines**  
    *  Query ↔ Document matching (web search, product search)
1. **Image–Text matching**  
    * Image ↔ Text similarity (Tiktok, Youtube search)
1. **Semantic text similarity**
    * entence ↔ Sentence matching (FAQ search, duplicate detection)

## Dot Product vs Cosine Similarity and Binary Cross-Entropy Loss

In this notebook, the similarity between user and movie embeddings is computed using the **dot product**.

Dot product between two embeddings it is called $z$ the similarity score (**logit**) :

$
z = u \cdot v = \sum_{i=1}^{d} u_i v_i \\
z = log(\frac{p}{1−p}​)
$



Where:

* $u$ = user embedding  
* $v$ = movie embedding  
* $d$ = embedding dimension   

This score $z$ is used directly as input to the Binary Cross-Entropy loss:

```python
loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
```

Since the model outputs raw scores (logits), TensorFlow internally applies the sigmoid function to convert scores into probabilities:

Sigmoid:

$
\sigma(z)=\frac{1}{1+e^{-z}}
$

Binary Cross-Entropy loss:

$
L = -\left[y\log(\sigma(z)) + (1-y)\log(1-\sigma(z))\right]
$

Where:

$y$ = true label (0 or 1)

$\sigma(z)$ = predicted probability

$L$ = loss

## Brief Discussion: Dot Product vs Cosine Similarity

Dot product depends on both vector direction and magnitude.

Cosine similarity depends only on direction, because vectors are normalized:

Cosine similarity:

$
\cos(\theta)=
\frac{u \cdot v}
{||u||,||v||}
$

If embeddings are normalized to unit length, the dot product becomes equivalent to cosine similarity.

* Dot product is commonly used during training because:
* it allows embedding magnitude to carry information
* it works naturally with logits and Binary Cross-Entropy
* it is computationally efficient

Cosine similarity is often used during retrieval or nearest-neighbor search, where only relative similarity matters.

**Important Note: Large Values Can Dominate Dot Product**

One limitation of dot product is that a single very large embedding value can dominate the similarity score.

Example:

If one dimension has a very large value:

$
u = [0.1,;0.2,;100]
$

then:

$
u \cdot v \approx 100 \times v_3
$

In this case:

* one dimension dominates the score
* other dimensions contribute very little
* the model may ignore useful features

Using cosine similarity (or normalized embeddings) reduces this risk because all dimensions are scaled equally.

## Prepare env

```sh
python3 -m venv ~/.venvs/two-tower-env

source ~/.venvs/two-tower-env/bin/activate

pip install --upgrade pip

pip install \
tensorflow \
tensorflow-datasets \
tensorflow-recommenders \
jupyter \
pandas \
matplotlib \
plotly 

pip install tensorflow tensorflow-datasets tensorflow-recommenders

pip install importlib-resources

# NOTE: optional if you are using vs code
pip install ipykernel

# NOTE: optional if you are using vs code
python -m ipykernel install \
--user \
--name twotower-env \
--display-name "Python (twotower)"

```

In [1]:
import os
# NOTE: tensorflow_recommenders still use keras 2 but tensorflow switch to keras 3
# setting keras 2 as keras version for tensorflow env
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_recommenders as tfrs

print("TF:", tf.__version__)
print("TFDS:", tfds.__version__)
print("TFRS:", tfrs.__version__)

TF: 2.16.2
TFDS: 4.9.9
TFRS: v0.7.7


In [2]:
import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Datasets: Movielens


* MovieLens is a public recommender dataset created by the GroupLens Research.
* It contains user ratings of movies, used to study recommendation algorithms.
* Popular sizes:
    * 100K — small, ideal for learning
    * 1M / 10M / 20M — research-scal

* Widely used for collaborative filtering

* Data

    * ratings table (user interactions) simplified


    | user_id | movie_id | rating | timestamp   |
    |--------|-----------|--------|-------------|
    | 196    | 242       | 3      | 881250949   |
    | 186    | 302       | 4      | 891717742   |
    | 22     | 377       | 1      | 878887116   |
    | 244    | 51        | 2      | 880606923   |
    | 166    | 346       | 5      | 886397596   |


    * movies table (item metadata) simplified


    | movie_id | movie_title        | genres               |
    |-----------|-------------------|----------------------|
    | 242       | Kolya (1996)      | Comedy,Drama         |
    | 302       | L.A. Confidential | Crime,Drama,Thriller |
    | 377       | Heavyweights      | Comedy               |
    | 51        | Legends of Fall   | Drama,Romance        |
    | 346       | Jackie Brown      | Crime,Drama          |

-----------

> PS: TensorFlow datasets are not pandas DataFrames. The behavior is more similar to Spark RDDs or Spark DataFrames in the sense that:
>
> - Data is processed as a **stream/pipeline**, not loaded fully into memory by default.
> - You **iterate over elements**, instead of indexing rows directly.
> - Operations are **lazy** — transformations are defined first, executed later.
> - There is no `.shape` like pandas; instead use:
>     - `tf.data.Dataset.cardinality(ds)` → number of elements
> - To inspect rows, use:
>     - `ds.take(n)` → similar to `head(n)`
> - Each row is usually a **dictionary of tensors**, not scalar values.
>
> Mental model:
>
> pandas → in-memory table  
> Spark → distributed lazy table  
> tf.data → streaming ML input pipeline
>
> Typical usage pattern:
>
> ds = tfds.load(...)
>
> ds = ds.shuffle(...)
>
> ds = ds.batch(...)
>
> ds = ds.prefetch(...)
>
> The stream data feeds efficiently the neural network during training.

In [3]:
import tensorflow_datasets as tfds

ratings = tfds.load(
    "movielens/100k-ratings",
    split="train"
)

movies = tfds.load(
    "movielens/100k-movies",
    split="train"
)

# Count rows
ratings_size = tf.data.Dataset.cardinality(ratings).numpy()
movies_size = tf.data.Dataset.cardinality(movies).numpy()

print(f"Ratings size: {ratings_size}")
print(f"Movies size: {movies_size}")

Ratings size: 100000
Movies size: 1682


In [4]:
import pandas as pd
print("Ratings sample:")
pd.DataFrame([{k: v.numpy() for k, v in x.items()} for x in ratings.take(3)])

print()
print("Movies sample:")
pd.DataFrame([{k: v.numpy() for k, v in x.items()} for x in movies.take(3)])

Ratings sample:


2026-03-29 17:12:35.701938: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-03-29 17:12:35.702355: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,bucketized_user_age,movie_genres,movie_id,movie_title,raw_user_age,timestamp,user_gender,user_id,user_occupation_label,user_occupation_text,user_rating,user_zip_code
0,45.0,[7],b'357',"b""One Flew Over the Cuckoo's Nest (1975)""",46.0,879024327,True,b'138',4,b'doctor',4.0,b'53211'
1,25.0,"[4, 14]",b'709',b'Strictly Ballroom (1992)',32.0,875654590,True,b'92',5,b'entertainment',2.0,b'80525'
2,18.0,[4],b'412',"b'Very Brady Sequel, A (1996)'",24.0,882075110,True,b'301',17,b'student',4.0,b'55439'



Movies sample:


2026-03-29 17:12:35.724960: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-03-29 17:12:35.725170: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,movie_genres,movie_id,movie_title
0,[4],b'1681',b'You So Crazy (1994)'
1,"[4, 7]",b'1457',b'Love Is All There Is (1996)'
2,"[1, 3]",b'500',b'Fly Away Home (1996)'


Explanations about vars

* bucket user age

| bucketized_user_age | Actual age range |
|--------------------|------------------|
| 1                  | Under 18         |
| 18                 | 18–24            |
| 25                 | 25–34            |
| 35                 | 35–44            |
| 45                 | 45–49            |
| 50                 | 50–55            |
| 56                 | 56+              |

* user_gender (is_male)
    * True = Male , False = Female

## Preprocessing

In [5]:
interactions = ratings.map(lambda x: {
    "user_id": x["user_id"],
    "user_age_bucket": x["bucketized_user_age"], # NOTE: age bucket 18–24
    "is_male_user": x["user_gender"],
    "movie_id": x["movie_id"],
    "rating": x["user_rating"]
})

In [7]:
def preprocess(x):

    label = tf.cast(x["rating"] >= 4, tf.float32)
    age_bucket = tf.cast(x["user_age_bucket"], tf.int32)
    is_male = tf.cast(x["is_male_user"], tf.float32)

    # NOTE: preparing data for fit: dict -> (features, label) format
    _row_tuple = (
        {
        "user_id": x["user_id"],
        "user_age_bucket": age_bucket,
        "is_male_user": is_male,
        "movie_id": x["movie_id"],
        },
        label
    )
    return _row_tuple

interactions_with_lables = interactions.map(preprocess)

pd.DataFrame([{k: v.numpy() for k, v in x[0].items()} | {"label": x[1].numpy()} for x in interactions_with_lables.take(3)])


2026-03-29 17:13:07.426349: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-03-29 17:13:07.426761: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,user_id,user_age_bucket,is_male_user,movie_id,label
0,b'138',45,1.0,b'357',1.0
1,b'92',25,1.0,b'709',0.0
2,b'301',18,1.0,b'412',1.0


In [8]:
import tensorflow as tf

tf.random.set_seed(42)

# NOTE: Shuffle for preventing bias split
interactions_with_lables = interactions_with_lables.shuffle(
    buffer_size=100_000,
    reshuffle_each_iteration=False
)

dataset_size = tf.data.Dataset.cardinality(
    interactions_with_lables
).numpy()

print("Dataset size:", dataset_size)

Dataset size: 100000


In [9]:
train_size = int(0.8 * dataset_size)

train = interactions_with_lables.take(train_size)
test  = interactions_with_lables.skip(train_size)

print("Train size:", train_size)
print("Test size:", dataset_size - train_size)

#pd.DataFrame([{k: v.numpy() for k, v in x.items()} for x in train.take(3)])
pd.DataFrame([{k: v.numpy() for k, v in x[0].items()} | {"label": x[1].numpy()} for x in train.take(3)])

Train size: 80000
Test size: 20000


2026-03-29 17:13:32.589976: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,user_id,user_age_bucket,is_male_user,movie_id,label
0,b'276',18,1.0,b'160',1.0
1,b'436',25,0.0,b'715',1.0
2,b'939',25,0.0,b'411',1.0


In [10]:
BATCH_SIZE = 256

train_batches = train.batch(BATCH_SIZE)
test_batches  = test.batch(BATCH_SIZE)

In [ ]:
# NOTE: prefetch for improving performance by overlapping data preprocessing and model execution
# train_batches = train_batches.prefetch(tf.data.AUTOTUNE)
# test_batches  = test_batches.prefetch(tf.data.AUTOTUNE)

In [13]:
for batch in train_batches.take(1):
   
   features, label = batch
   for k, v in features.items():
       print(f"{k} shape: {v.shape}; dtype: {v.dtype}")

print(f"label shape: {label.shape}, label dtype: {label.dtype}")


user_id shape: (256,); dtype: <dtype: 'string'>
user_age_bucket shape: (256,); dtype: <dtype: 'int32'>
is_male_user shape: (256,); dtype: <dtype: 'float32'>
movie_id shape: (256,); dtype: <dtype: 'string'>
label shape: (256,), label dtype: <dtype: 'float32'>


2026-03-29 17:17:10.198036: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Summary

* checking positive and negative cases for imbalancing

In [52]:
import tensorflow as tf

def count_labels(dataset):

    pos = 0
    neg = 0

    for X, labels in dataset:
        # labels = x["label"]

        pos += tf.reduce_sum(labels).numpy()
        neg += tf.reduce_sum(1 - labels).numpy()

    total = pos + neg

    print("Total:", int(total))
    print("Positive:", int(pos))
    print("Negative:", int(neg))

    print("Positive %:", round(pos / total * 100, 2))
    print("Negative %:", round(neg / total * 100, 2))

    return pos, neg

print("TRAIN distribution")
train_pos, train_neg = count_labels(train)

print("\nTEST distribution")
test_pos, test_neg = count_labels(test)

TRAIN distribution


2026-03-29 17:55:57.471996: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Total: 80000
Positive: 44355
Negative: 35645
Positive %: 55.44
Negative %: 44.56

TEST distribution
Total: 20000
Positive: 11020
Negative: 8980
Positive %: 55.1
Negative %: 44.9


2026-03-29 17:56:02.577211: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Define two towers arch

* inputs -> feature encoding -> dense representation -> final embedding
* user embedding · movie embedding = score 
    * good pair → higher dot product
    * bad pair → lower dot product

In [61]:
EMBEDDING_DIM = 10

### User tower

* **TODO**:
    * Still need to add iteration features

```
                         User Tower

   user_id              user_age_bucket              is_male
  (string)                  (int)                    (float)
     |                        |                         |
     v                        v                         v
+-------------+        +---------------+                |
| StringLookup|        | IntegerLookup |                |
+-------------+        +---------------+                |
     |                        |                         |
     v                        v                         |
+-------------+        +---------------+                |
| Embedding   |        | Embedding     |                |
| (EMBED_DIM) |        | (dim=8)       |                |
+-------------+        +---------------+                |
     |                        |                         |
     +------------------------+-------------------------+
                              |
                              v
                     +-------------------+
                     |   Concatenate     |
                     | (batch, D + 8 + 1)|
                     +-------------------+
                              |
                              v
                     +-----------------------+
                     |  Dense(20, relu) + L2 |
                     +-----------------------+
                              |
                              v
                     +-----------------------+
                     |     Dropout(0.35)     |
                     +-----------------------+
                              |
                              v
                     +-----------------------+
                     |  Dense(EMBEDDING_DIM) |
                     +-----------------------+
                              |
                              v
                     +-----------------------+
                     |    User Embedding     | ───> dot product with Movie Embedding
                     |   (EMBEDDING_DIM,)    |
                     +-----------------------+                                        
```

In [ ]:
import tensorflow as tf

class UserEmbeddingBlock(tf.keras.Model):

    def __init__(self, users, age_vocab, embedding_dim):

        super().__init__()

        # encoders
        self.user_id_encoder = tf.keras.layers.StringLookup(
            vocabulary=users,
            mask_token=None
        )

        self.age_encoder = tf.keras.layers.IntegerLookup(
            vocabulary=age_vocab,
            mask_token=None
        )

        # embeddings
        self.user_id_embedding = tf.keras.layers.Embedding(
            input_dim=len(users) + 1,
            output_dim=embedding_dim
        )

        self.age_embedding = tf.keras.layers.Embedding(
            input_dim=len(age_vocab) + 1,
            output_dim=8
        )

    def call(self, X):

        user_id = X["user_id"]
        user_age = tf.cast(X["user_age_bucket"], tf.int32)
        user_gender = X["is_male_user"]

        # encode
        user_id_idx = self.user_id_encoder(user_id)
        age_idx = self.age_encoder(user_age)

        # internal user id embeddings: shape (batch,) -> (batch, embedding_dim)
        user_emb = self.user_id_embedding(user_id_idx)
        # internal age embeddings: shape (batch,) -> (batch, 8)
        age_emb = self.age_embedding(age_idx)

        # reshape gender: shape: (batch,) -> (batch, 1)
        user_gender = tf.expand_dims(user_gender, axis=-1)

        # [debug]
        # tf.print("user_emb shape:", tf.shape(user_emb))
        # tf.print("age_emb shape:", tf.shape(age_emb))
        # tf.print("user_gender shape:", tf.shape(user_gender))

        # concat all internal embeddings in one single tensor
        x = tf.concat([user_emb, age_emb, user_gender],axis=1)

        return x


class UserTower(tf.keras.Model):

    def __init__(self, users, age_vocab, embedding_dim=EMBEDDING_DIM):

        super().__init__()

        self.unique_users = users
        self._n_unique_users = len(users)

        self.unique_ages = age_vocab
        self._n_unique_ages = len(age_vocab)

        self.embedding_dim = embedding_dim

        self.dnn = None
        self._build_dnn()

    def _build_dnn(self) -> None:

        # internal embedding layers
        self.embedding_layers = UserEmbeddingBlock(
            self.unique_users,
            self.unique_ages,
            self.embedding_dim
        )

        # combine all user features
        self.dense_1 = tf.keras.layers.Dense(20, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(2e-5))
        self.dropout = tf.keras.layers.Dropout(0.35)
        self.dense_2 = tf.keras.layers.Dense(self.embedding_dim)

        # keep your dnn block idea for the final dense part
        self.dnn = tf.keras.Sequential([
            self.embedding_layers,
            self.dense_1,
            self.dropout,
            self.dense_2   # final user embedding
        ])

    def call(self, X):

        y = self.dnn(X)

        return y

    def predict_embedding(self, X):

        users_embedding = self.call(X)

        return users_embedding

### Movie towers

```
                                    Movie Tower
                                                                                      
  movie_id                                                                            
  (string)                                                                            
     |                                                                                
     v                                                                                
+----------------------+                                                             
| StringLookup         |                                                                                                                        
+----------------------+                                                             
     |                                                                                
     v                                                                                
+----------------------+                                                             
| Embedding            |                                                                                                                       
| shape: (batch, D)    |                                                             
+----------------------+                                                             
     |                                                                                
     v                                                                                
+-----------------------+                                                            
|  Dense(20, relu) + L2 |                                                            
+-----------------------+                                                            
     |                                                                                
     v                                                                                
+-----------------------+                                                            
|     Dropout(0.35)     |                                                            
+-----------------------+                                                            
     |                                                                                
     v                                                                                
+-----------------------+                                                            
|  Dense(EMBEDDING_DIM) |                                                            
+-----------------------+                                                            
     |                                                                                
     v                                                                                
+-----------------------+                                                            
|   Movie Embedding     |  ──────> dot product with User Embedding                  
|   (EMBEDDING_DIM,)    |                                                            
+-----------------------+                                                            
```

In [63]:
class MovieEmbeddingBlock(tf.keras.Model):

    def __init__(self, movies, embedding_dim):

        super().__init__()

        # encoder
        self.movie_id_encoder = tf.keras.layers.StringLookup(
            vocabulary=movies,
            mask_token=None
        )

        # internal movies embedding
        self.movie_id_embedding = tf.keras.layers.Embedding(
            input_dim=len(movies) + 1,
            output_dim=embedding_dim
        )

    def call(self, X):

        movie_id = X["movie_id"]

        # encode
        movie_id_idx = self.movie_id_encoder(movie_id)

        # internal movie embeddings: shape (batch,) -> (batch, embedding_dim)
        movie_emb = self.movie_id_embedding(movie_id_idx)

        return movie_emb
    
class MovieTower(tf.keras.Model):

    def __init__(self, movies, embedding_dim=EMBEDDING_DIM):

        super().__init__()

        self.unique_movies = movies
        self._n_unique_movies = len(movies)

        self.embedding_dim = embedding_dim

        self.dnn = None
        self._build_dnn()

    def _build_dnn(self) -> None:

        # internal embedding layers
        self.embedding_layers = MovieEmbeddingBlock(
            self.unique_movies,
            self.embedding_dim
        )

        # combine all movie features
        self.dense_1 = tf.keras.layers.Dense(20, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(2e-5))
        self.dropout = tf.keras.layers.Dropout(0.35)
        self.dense_2 = tf.keras.layers.Dense(self.embedding_dim)

        self.dnn = tf.keras.Sequential([
            self.embedding_layers,
            self.dense_1,
            self.dropout,
            self.dense_2   # final movie embedding
        ])

    def call(self, X):

        y = self.dnn(X)

        return y

    def predict_embedding(self, X):

        movie_embedding = self.call(X)

        return movie_embedding

### 2 towers

```
                 User Tower                         Movie Tower

          user_id     age    is_male                  movie_id
             |        |        |                         |
             v        v        v                         v
          StrLkp    IntLkp     |                      StrLkp
             |        |        |                         |
             v        v        |                         v
          Emb(D)    Emb(8)     |                       Emb(D)
             |        |        |                         |
             +--------+--------+                         |
                      |                                  |
              Concat(D + 8 + 1)                    Concat(D)
                      |                                  |
               Dense(20, relu)                    Dense(20, relu)
                      |                                  |
                Dropout(0.35)                      Dropout(0.35)
                      |                                  |
                  Dense(D)                           Dense(D)
                      |                                  |
                 user_emb (D)                      movie_emb (D)
                      |                                  |
                      +------------> dot <---------------+
                                      |
                                  score (logit)
                                      |
                    BinaryCrossentropy(from_logits=True)
```

In [64]:
import tensorflow as tf


class TwoTowerModel(tf.keras.Model):

    def __init__(self, user_tower, movie_tower):
        
        super().__init__()
        self.user_tower = user_tower
        self.movie_tower = movie_tower

    def call(self, X):
        user_embedding = self.user_tower({
            "user_id": X["user_id"],
            "user_age_bucket": X["user_age_bucket"],
            "is_male_user": X["is_male_user"],
        })

        movie_embedding = self.movie_tower({
            "movie_id": X["movie_id"],
        })

        # logit
        score = tf.reduce_sum(user_embedding * movie_embedding, axis=1)
        return score

### Generating vocabularies

In [65]:
# NOTE: buildong vocabularies: list of unique users and mvies and ages for the embedding layers

unique_user_ids = set()
unique_ages = set()
unique_movie_ids = set()

for X, y in train:

    unique_user_ids.add(
        X["user_id"].numpy().decode("utf-8")
    )

    unique_movie_ids.add(
        X["movie_id"].numpy().decode("utf-8")
    )

    unique_ages.add(
        int(X["user_age_bucket"].numpy())
    )

# convert to sorted lists (important for reproducibility)

user_vocab = sorted(unique_user_ids)
age_vocab = sorted(unique_ages)
movie_vocab = sorted(unique_movie_ids)

print("n_users:", len(user_vocab))
print("n_ages:", len(age_vocab))
print("n_movies:", len(movie_vocab))

n_users: 943
n_ages: 7
n_movies: 1648


2026-03-29 21:28:40.817372: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### Inspecting the embeddings examples

In [66]:
# NOTE: checking

user_tower = UserTower(
    users=user_vocab,
    age_vocab=age_vocab,
    embedding_dim=EMBEDDING_DIM
)

movie_tower = MovieTower(
    movies=movie_vocab,
    embedding_dim=EMBEDDING_DIM
)

In [67]:
# One sample batch (batch size = 1)

user_batch = {
    "user_id": tf.constant(["138"]),
    "user_age_bucket": tf.constant([45], dtype=tf.int32),
    "is_male_user": tf.constant([1.0], dtype=tf.float32),
}

movie_batch = {
    "movie_id": tf.constant(["357"])
}


user_embedding = user_tower(user_batch)

print("User embedding shape:")
print(user_embedding.shape)


movie_embedding = movie_tower(movie_batch)
print("Movie embedding shape:")
print(movie_embedding.shape)

score1 = tf.reduce_sum(user_embedding * movie_embedding, axis=1)
print("Score shape:")
print(score1.shape)  
print("Score value:")
print(score1.numpy())

User embedding shape:
(1, 10)
Movie embedding shape:
(1, 10)
Score shape:
(1,)
Score value:
[0.00215989]


In [68]:
two_tower = TwoTowerModel(
    user_tower=user_tower,
    movie_tower=movie_tower
)

# NOTE: concat dicts for user and movie features
X_batch = user_batch | movie_batch

score2  = two_tower.call(X_batch)

print("Score shape:")
print(score2.shape)
print("Score value:")
print(score2.numpy())


assert score1.numpy() == score2.numpy(), "Scores from manual and two tower model should be the same"

Score shape:
(1,)
Score value:
[0.00215989]


## Training the towers

In [69]:
print(f"unique useres: {len(user_vocab)}")
print(f"unique age buckets: {len(age_vocab)}")
print(f"unique movies: {len(movie_vocab)}")

print(f"train size: {train_size}")
print(f"test size: {dataset_size - train_size}")
print(f"Embedding dim: {EMBEDDING_DIM}")

user_tower = UserTower(
    users=user_vocab,
    age_vocab=age_vocab,
    embedding_dim=EMBEDDING_DIM
)

movie_tower = MovieTower(
    movies=movie_vocab,
    embedding_dim=EMBEDDING_DIM
)

two_tower_model = TwoTowerModel(user_tower, movie_tower)

two_tower_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(threshold=0.0, name="binary_acc"),
    ],
)

unique useres: 943
unique age buckets: 7
unique movies: 1648
train size: 80000
test size: 20000
Embedding dim: 10


In [70]:
history = two_tower_model.fit(
    train_batches,
    validation_data=test_batches,
    epochs=10,
)

Epoch 1/10
313/313 [==============================] - 4s 7ms/step - loss: 0.6426 - binary_acc: 0.6202 - val_loss: 0.5823 - val_binary_acc: 0.6938
Epoch 2/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5754 - binary_acc: 0.7021 - val_loss: 0.5706 - val_binary_acc: 0.7053
Epoch 3/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5612 - binary_acc: 0.7131 - val_loss: 0.5659 - val_binary_acc: 0.7079
Epoch 4/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5536 - binary_acc: 0.7182 - val_loss: 0.5633 - val_binary_acc: 0.7098
Epoch 5/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5476 - binary_acc: 0.7190 - val_loss: 0.5636 - val_binary_acc: 0.7103
Epoch 6/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5445 - binary_acc: 0.7211 - val_loss: 0.5634 - val_binary_acc: 0.7117
Epoch 7/10
313/313 [==============================] - 3s 6ms/step - loss: 0.5416 - binary_acc: 0.7218 - val_loss: 0.5637 - v

## Evaluate

In [71]:
import plotly.graph_objects as go

def plot_learning_cuve(loss, val_loss):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=loss,
        mode="lines+markers",
        name="Train Loss"
    ))

    fig.add_trace(go.Scatter(
        y=val_loss,
        mode="lines+markers",
        name="Validation Loss"
    ))

    fig.update_layout(
        title="Learning Curve — Loss",
        xaxis_title="Epoch",
        yaxis_title="Binary Crossentropy Loss",
        template="plotly_white"
    )

    fig.show()


def plot_accuracy_curve(acc, val_acc):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=acc,
        mode="lines+markers",
        name="Train Accuracy"
    ))

    fig.add_trace(go.Scatter(
        y=val_acc,
        mode="lines+markers",
        name="Validation Accuracy"
    ))

    fig.update_layout(
        title="Learning Curve — Accuracy",
        xaxis_title="Epoch",
        yaxis_title="Accuracy",
        template="plotly_white"
    )

    fig.show()


history.history.keys()
history_dict = history.history
print()
plot_learning_cuve(history_dict["loss"], history_dict["val_loss"])
print()
plot_accuracy_curve(history_dict["binary_acc"], history_dict["val_binary_acc"])

dict_keys(['loss', 'binary_acc', 'val_loss', 'val_binary_acc'])

## Good and bads distributions

In [72]:
import numpy as np

scores = []
labels = []

for features, y in test_batches:

    preds = two_tower_model(features)

    scores.extend(preds.numpy())
    labels.extend(y.numpy())

scores = np.array(scores)
labels = np.array(labels)

print("Total samples:", len(scores))

Total samples: 20000


2026-03-29 21:29:16.897386: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [73]:
good_scores = scores[labels == 1]
bad_scores  = scores[labels == 0]

print("Good samples:", len(good_scores))
print("Bad samples:", len(bad_scores))

Good samples: 11020
Bad samples: 8980


In [74]:
import plotly.graph_objects as go

def plot_good_and_bads(good_scores, bad_scores):

    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=good_scores,
        name="Good (label=1)",
        opacity=0.6,
        nbinsx=50
    ))

    fig.add_trace(go.Histogram(
        x=bad_scores,
        name="Bad (label=0)",
        opacity=0.6,
        nbinsx=50
    ))

    fig.update_layout(
        title="Score Distribution — Good vs Bad",
        xaxis_title="Model Score (logit)",
        yaxis_title="Count",
        barmode="overlay",
        template="plotly_white"
    )

    fig.show()

plot_good_and_bads(good_scores, bad_scores)